## Introduction to `torch.autograd`

Based on https://docs.pytorch.org/tutorials/beginner/blitz/autograd_tutorial.html

`torch.autograd` is PyTorch's automatic different engine used in neural network training.

In the forward prop, the NN runs the input data through its functions, running computations through the parameters and outputs the propagation.

In the backward propagation, then NN adjusts the parameters proportionate to the error in its guess.
It does this by traversing backwards from the output, collecting the derivatives of the error with respect to the parameters of the functions (gradients), and optimizing the parameters using gradient descent.
Check out the [backprop video from 3Blue1Brown](https://www.youtube.com/watch?v=tIeHLnjs5U8)

## Usage in PyTorch

Let’s take a look at a single training step. For this example, we load a pretrained resnet18 model from torchvision. We create a random data tensor to represent a single image with 3 channels, and height & width of 64, and its corresponding label initialized to some random values. Label in pretrained models has shape (1,1000).

In [2]:
import torch
from torchvision.models import resnet18, ResNet18_Weights

In [5]:
model = resnet18(weights=ResNet18_Weights.DEFAULT)
data = torch.rand(1, 3, 64, 64)
labels = torch.rand(1, 1000)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\clhabins/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100.0%


In [6]:
prediction = model(data) #forward pass

In [11]:
# Use model's prediction and corresponding label to compute error loss.
# Then backpropagate the error through the network, by calling loss.backward()
# Autograd then calculates and stores the gradients for each model parameter in the parameter's .grad attribute.

loss = (prediction - labels).sum()
loss.backward() # backward pass

In [20]:
# Next, load an optimizer, in this case SGD with learning rate of 0.01 and momentum of 0.9.
# We register all the parameters of the model in the optimizer
optim = torch.optim.SGD(model.parameters(), lr=1e-2, momentum=0.9)

In [ ]:
# Finally, we call step() to initiate gradient descent. The optimized adjusts each parameter by its gradient stored in .grad

optim.step() # gradient descent

## Differentation in Autograd

Let's take a look at how `autograd` collects gradients.

In [ ]:
# Create tensors with requires_grad=True. This signals to autograd that every operation on them should be tracked.

a = torch.tensor([2., 3.], requires_grad=True)
b = torch.tensor([6., 4.], requires_grad=True)


In [25]:
# Create another tensor from a and b

Q = 3*a**3 - b**2

Let's assume a and b are parameters of an NN, and Q to be the error.
In NN training, we want gradients of the error w.r.t parameters
(using d to represent the partial derivative operator):

dQ/da = 3 * (3a^2) = 9a^2 (this is a partial derivative where b is treated like a constant)

dQ/db = -(2b) = -2b

When we call `Q.backward()`, autograd calcualtes the gradients and stores them in the respective tensors `.grad` attribute.

We need to explicitly pass gradient argument in `Q.backward()` because it's a vector (grad can be implicitly created only for scalar outputs).
`gradient` is a tensor of the same shape as Q and it represents
the gradient of Q w.r.t. to itself.
i.e.

dQ/dQ = 1

In [27]:
external_grad = torch.tensor([1., 1.])
Q.backward(gradient=external_grad)

In [30]:
# Gradients are not collected in a.grad and b.grad
print("a.grad:", a.grad, "expected:", 9*a**2, "correct =", a.grad == 9*a**2)
print("b.grad:", b.grad, "expected:", -2*b, "correct =", b.grad == -2*b)

a.grad: tensor([36., 81.]) expected: tensor([36., 81.], grad_fn=<MulBackward0>) correct = tensor([True, True])
b.grad: tensor([-12.,  -8.]) expected: tensor([-12.,  -8.], grad_fn=<MulBackward0>) correct = tensor([True, True])


## Vector calculus using autograd

If you have a vector-valued function y = f(x) (where y and x are vectors),
then the gradient of y with respect to x is a [Jacobian matrix](https://en.wikipedia.org/wiki/Jacobian_matrix_and_determinant) J.
The Jacobian matrix of a vector-valued function of several variables is the matrix of all its first-order partial derivatives.

*Note*: I'm using d as the partial derivative operator

```
J = (dy/dx_1 ... dy/dx_n) =


[
    dy_1/dx_1, ... dy_1/dx_n
    ...
    dy_m/dx_1, ... dy_m/dx_n
]
```

*Note* that y and x can have different lengths.

Generally speaking, `torch.autograd` is an eingine for computing vector-Jacobian product. That is, given any vector v, compute the product `J.T dot v` (where .T is transpose operation)

If v happens to be gradient of a scalar function l = g(y):

```
v = (dl/dy_1 ... dl/dy_m).T
```

then by chain rule, the vector-Jacobian product would be the gradient of l with respect to x:

```
J.T * v = 

[
  dy_1/d_x1 ... dy_m/dx_1
  ...
  dy_1/dx_n ... dy_m/dx_n
]
*
[
  dl/dy_1
  ...
  dl/dy_m
]
=
[
  dl/dx_1
  ...
  dl/dx_n
] 
```


This characteristic of vector-Jacobian product is what we use in the above example; `external_grad` represents v.

## Computational graph

Conceptually, autograd keeps a record of data (tensors) & all executed operations (along with the resulting new tensors) in a directed acyclic graph (DAG) consisting of Function objects. In this DAG, leaves are the input tensors, roots are the output tensors. By tracing this graph from roots to leaves, you can automatically compute the gradients using the chain rule.

In a forward pass, autograd does two things simultaneously:

- run the requested operation to compute a resulting tensor, and
- maintain the operation’s gradient function in the DAG.

The backward pass kicks off when .backward() is called on the DAG root. autograd then:

- computes the gradients from each .grad_fn,
- accumulates them in the respective tensor’s .grad attribute, and
- using the chain rule, propagates all the way to the leaf tensors.

**Note**: DAGs are dynamic in PyTorch An important thing to note is that the graph is recreated from scratch; after each .backward() call, autograd starts populating a new graph. This is exactly what allows you to use control flow statements in your model; you can change the shape, size and operations at every iteration if needed.

## Exclusion from the DAG

`torch.autograd` tracks operations on all tensors which have their `requires_grad` flag set to True. For tensors that don’t require gradients, setting this attribute to False excludes it from the gradient computation DAG.

The output tensor of an operation will require gradients even if only a single input tensor has `requires_grad=True`.

In [31]:
x = torch.rand(5, 5)
y = torch.rand(5, 5)
z = torch.rand((5, 5), requires_grad=True)

a = x + y
print(f"Does `a` require gradients?: {a.requires_grad}")
b = x + z
print(f"Does `b` require gradients?: {b.requires_grad}")

Does `a` require gradients?: False
Does `b` require gradients?: True


In a NN, parameters that don’t compute gradients are usually called **frozen parameters**.

It is useful to “freeze” part of your model if you know in advance that you won’t need the gradients of those parameters
(this offers some performance benefits by reducing autograd computations).

In finetuning, we freeze most of the model and typically only modify the classifier layers to make predictions on new labels.

Let’s walk through a small example to demonstrate this. As before, we load a pretrained resnet18 model, and freeze all the parameters.

In [32]:
from torch import nn, optim

model = resnet18(weights=ResNet18_Weights.DEFAULT)

# Freeze all the parameters in the network
for param in model.parameters():
    param.requires_grad = False

In [33]:
# Let’s say we want to finetune the model on a new dataset with 10 labels.
# In resnet, the classifier is the last linear layer model.fc.
# We can simply replace it with a new linear layer (unfrozen by default) that acts as our classifier.

model.fc = nn.Linear(512, 10)

In [34]:
# Now all parameters in the model, except the parameters of model.fc, are frozen.
# The only parameters that compute gradients are the weights and bias of model.fc.

# Optimize only the classifier
optimizer = optim.SGD(model.parameters(), lr=1e-2, momentum=0.9)

Notice although we register all the parameters in the optimizer, the only parameters that are computing gradients (and hence updated in gradient descent) are the weights and bias of the classifier.

The same exclusionary functionality is available as a context manager in [`torch.no_grad()`](https://pytorch.org/docs/stable/generated/torch.no_grad.html)